SummarizationMiddleware 摘要中间件
HumanInTheLoopMiddleware：人工介入中间件
ToolCallLimitMiddleware：限制工具调用次数中间件
....

基类：AgentMiddleware：所有中间件都继承该类，如果要自定义中间件则继承这类


In [ ]:
import datetime
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, before_model, after_model, after_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.prebuilt.tool_node import ToolCallRequest
from rich import print as rprint

load_dotenv(verbose=True)

api_key = os.getenv("COMPANY_API_KEY")
base_url = os.getenv("COMPANY_BASE_URL")

llm = init_chat_model(
    model="glm-5.3-flash",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)


prompt  = """
    你是一个通用助手
"""

messages = [
    SystemMessage(content="你是一个专业的 LangChain 助手，回答简洁、准确、适合初学者。"),

    # 第 1 轮
    HumanMessage(content="你好，能简单介绍一下 LangChain 吗？"),
    AIMessage(content="LangChain 是一个用于构建大语言模型应用的框架，提供模型、提示、链、记忆、检索、代理等组件。"),

    # 第 2 轮
    HumanMessage(content="LangChain 里的消息对象有哪些类型？"),
    AIMessage(content="常见的有 SystemMessage、HumanMessage、AIMessage、ToolMessage、FunctionMessage、ChatMessage 等。"),

    # 第 3 轮
    HumanMessage(content="SystemMessage 和 HumanMessage 有什么区别？"),
    AIMessage(content="SystemMessage 用于设定系统指令或角色，HumanMessage 表示用户输入。"),

    # 第 4 轮
    HumanMessage(content="AIMessage 可以包含工具调用吗？"),
    AIMessage(content="可以。AIMessage 可以通过 tool_calls 字段表示模型请求调用工具。"),

    # 第 5 轮
    HumanMessage(content="怎么把消息列表传给模型？"),
    AIMessage(content="先构建 messages 列表，然后调用 model.invoke(messages)，例如 ChatOpenAI 的 invoke 方法。"),

    # 第 6 轮
    HumanMessage(content="如果要做多轮对话，需要保存历史吗？"),
    AIMessage(content="需要。把历史 HumanMessage 和 AIMessage 按顺序加入消息列表，或者使用 LangChain 的记忆组件。"),

    # 第 7 轮
    HumanMessage(content="LangChain 的 LCEL 是什么？"),
    AIMessage(content="LCEL 是 LangChain Expression Language，可以用管道符组合组件，例如 prompt | model | parser。"),

    # 第 8 轮
    HumanMessage(content="RAG 在 LangChain 中怎么实现？"),
    AIMessage(content="通常用文档加载器、文本分割器、向量存储、检索器和模型链组合实现 RAG。"),

    # 第 9 轮
    HumanMessage(content="Agent 和 Chain 有什么区别？"),
    AIMessage(content="Chain 通常是固定流程，Agent 则由模型动态决定调用哪些工具以及执行步骤。"),

    # 第 10 轮
    HumanMessage(content="最后，给我一个学习 LangChain 的建议吧。"),
    AIMessage(content="先掌握消息对象、提示模板和 LCEL，再做 RAG 和 Agent 小项目，边写边查官方文档。"),
]

summary_prompt = """你是对话摘要压缩器。请把下面的历史对话压缩成极简摘要，供另一个模型继续对话使用。

规则：
1. 只保留：用户目标、已确认事实、关键结论、重要决定、待办事项、用户偏好、专有名词、代码、API、参数、数字、错误信息。
2. 删除：寒暄、重复、客套、解释过程、无关细节。
3. 不编造、不推测、不添加原文没有的信息。
4. 默认用中文，不要标题、不要分节，直接一段话或最多 5 条短句。
5. 摘要必须明显短于原文；如果原文很短，可以只保留一句话。

历史对话：
{messages}
"""


graph = create_agent(
    model=llm,
    # system_prompt= prompt,
    # 给智能体起个名字
    name="通用智能体",
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens",200),
            keep=("messages",2), # 保留2条
            summary_prompt=summary_prompt) # 自定义摘要提示词
    ]
)

response = graph.invoke({
    "messages": messages
})

rprint(response)






案例：人工审核的邮件发送智能体


In [ ]:
import os
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from dotenv import load_dotenv


load_dotenv(verbose=True)

api_key = os.getenv("COMPANY_API_KEY")
base_url = os.getenv("COMPANY_BASE_URL")

# 初始化模型（以 OpenAI 为例，也可替换为其他兼容模型）
llm = init_chat_model(
    # model="deepseek-v4-pro",
    model="glm-5.3-flash",
    # glm-5.3-flash
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """发送邮件到指定地址。"""
    # 实际场景中这里会连接 SMTP 服务发送邮件
    return f"邮件已发送至 {to}，主题：{subject}"

@tool
def read_calendar(date: str) -> str:
    """读取指定日期的日历事件。"""
    return f"{date} 的日历事件：10:00 团队会议"


# agent = create_agent(
#         model=llm,
#         tools=[send_email, read_calendar]
#     )
#
# response = agent.invoke({
#     "messages": "你是谁？"
# })
#
# print(response)

# 必须使用 checkpointer 来保存中断状态，否则无法恢复执行[reference:2]
checkpointer = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=[send_email, read_calendar],
    system_prompt="你是一个邮件助手，可以帮助用户发送邮件和查看日历。",
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # send_email 需要审核，允许所有决策类型
                "send_email": True,
                # read_calendar 安全，无需审核
                "read_calendar": False,
            }
        )
    ],
    checkpointer=checkpointer,  # 必填
)



# 配置线程 ID，用于恢复时定位同一个会话
config = {"configurable": {"thread_id": "email-thread-1"}}

# 用户请求发送邮件
result = agent.invoke(
    {"messages": [{"role": "user", "content": "帮我给 boss@company.com 发一封邮件，主题是'周报'，内容是'本周工作已完成。'"}]},

    config=config
)

# 检查是否触发了中断
if "__interrupt__" in result:
    interrupt_request = result["__interrupt__"][0]
    print("触发了人工审核中断：")
    print(interrupt_request.value)  # 包含待审核的工具调用信息
    # 再次调用
    result = agent.invoke(

    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config)
    rprint(result)

else:
    print("没有中断，直接执行完成：", result["messages"][-1].content)




In [ ]:
import os
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv(verbose=True)

api_key = os.getenv("COMPANY_API_KEY")
base_url = os.getenv("COMPANY_BASE_URL")

# 初始化模型（以 OpenAI 为例，也可替换为其他兼容模型）
llm = init_chat_model(
    # model="deepseek-v4-pro",
    model="glm-5.3-flash",
    # glm-5.3-flash
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)


agent = create_agent(
    model=llm,
    # system_prompt="你是一个邮件助手，可以帮助用户发送邮件和查看日历。",
    middleware=[

        TodoListMiddleware()
    ],
)

config = {"configurable": {"thread_id": "trip-1"}}

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "帮我调研杭州国庆期间的天气、高铁余票、热门酒店价格，然后综合这些信息给我一份 7 天行程建议"}]},
    # config=config,
    stream_mode="updates",   # ← 关键：每步输出增量
):
    # chunk 是 dict: {节点名: {状态字段: 新值}}
    print("=== chunk ===")
    print(chunk)

    # 只看 todos 变更
    if "todos" in str(chunk):
        for node, updates in chunk.items():
            if isinstance(updates, dict) and "todos" in updates:
                print(f"[{node}] todos 更新：")
                for todo in updates["todos"]:
                    status_icon = {
                        "pending": "⏳",
                        "in_progress": "🔄",
                        "completed": "✅",
                    }.get(todo["status"], "❓")
                    print(f"  {status_icon} {todo['content']} — {todo['status']}")


# response = agent.invoke({
#     # 帮我调研杭州国庆期间的天气、高铁余票、热门酒店价格，然后综合这些信息给我一份 7 天行程建议
#     "messages": [{"role":"user","content":"帮我调研杭州国庆期间的天气、高铁余票、热门酒店价格，然后综合这些信息给我一份 7 天行程建议"}]
#     # "messages": [{"role":"user","content":"hi"}]
# })

rprint(response)



In [ ]:
from typing import Any
from langgraph.runtime import Runtime
import os
from langchain.agents import create_agent, AgentState

from langchain_core.messages import SystemMessage,AIMessage
from langchain.agents.middleware import before_model,after_model,after_agent,before_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint
import datetime

current_time = datetime.datetime.now().isoformat()

load_dotenv(verbose=True)

api_key = os.getenv("COMPANY_API_KEY")
base_url = os.getenv("COMPANY_BASE_URL")

# 初始化模型（以 OpenAI 为例，也可替换为其他兼容模型）
llm = init_chat_model(
    # model="deepseek-v4-pro",
    model="glm-5.3-flash",
    # glm-5.3-flash
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)

@before_agent
def add_agnet_preHandler(state:AgentState,runtime:Runtime) -> dict[str, Any] | None:
    print("agent pre handler")


@after_agent
def add_agent_postHandler(state:AgentState,runtime:Runtime) -> dict[str, Any] | None:
    print("agent post handler")

# 自定义中间件：
@before_model
def add_system_time(state:AgentState,runtime:Runtime) -> dict:
    """每次调用模型之前都会注入当前系统时间"""
    current_time = datetime.datetime.now().isoformat()
    print("当前时间：" + current_time)
    # 返回值会合并到Agent的状态中？这个Agent的状态是什么？
    return {
        "messages":[SystemMessage(content=f"当前系统时间：{current_time}")]
    }


@after_model
def log_after_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """每次模型调用完成后打印日志。"""
    # 取最后一条消息，就是模型刚返回的 AIMessage
    last_msg = state["messages"][-1]

    # 安全判断：确保是 AIMessage
    if isinstance(last_msg, AIMessage):
        content_preview = (last_msg.content or "")[:80].replace("\n", " ")
        tool_calls = [tc["name"] for tc in (last_msg.tool_calls or [])]
        usage = last_msg.usage_metadata or {}

        print("=" * 60)
        print(f"[模型调用完成]")
        print(f"  内容预览: {content_preview!r}")
        if tool_calls:
            print(f"  工具调用: {tool_calls}")
        if usage:
            print(f"  Token用量: input={usage.get('input_tokens')}, "
                  f"output={usage.get('output_tokens')}, "
                  f"total={usage.get('total_tokens')}")
        print("=" * 60)

    # 返回 None 表示不修改状态
    return None



agent = create_agent(
    model=llm,
    # system_prompt="你是一个邮件助手，可以帮助用户发送邮件和查看日历。",
    middleware=[
        add_system_time,
        log_after_model,
        add_agnet_preHandler,
        add_agent_postHandler
    ],
)


response = agent.invoke(
    {
        "messages": [{"role": "user", "content": "你好"}]
    }
)

rprint(response)




自定义中间件：

中间件有两种类型：节点式和包裹式。
节点式：在不同节点执行不同的hook函数，比如模型执行前，模型执行后
   - @after_model
   - @after_agent
   - @before_model
   - @before_agent

包裹式：在模型、工具执行之前，以及执行之后，拿到handler可以决定是否调用
- @wrap_model_call
- @wrap_tool_call



具体的定义同样有两种实现方式：
1. 基于装饰器
2. 基于类




In [ ]:
from typing import Any, Callable
from langgraph.runtime import Runtime
import os
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.messages import SystemMessage, AIMessage, ToolMessage
from langchain.agents.middleware import before_model,after_model
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint
import datetime

current_time = datetime.datetime.now().isoformat()

load_dotenv(verbose=True)

api_key = os.getenv("COMPANY_API_KEY")
base_url = os.getenv("COMPANY_BASE_URL")

# 初始化模型（以 OpenAI 为例，也可替换为其他兼容模型）
llm = init_chat_model(
    # model="deepseek-v4-pro",
    model="glm-5.3-flash",
    # glm-5.3-flash
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)

@wrap_tool_call
def wrap_tool_call_middleware(
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]]
    ) -> ToolMessage | Command[Any]:
    handler
    pass




agent = create_agent(
    model=llm,
    middleware=[
        wrap_tool_call_middleware
    ],
)






